In [ ]:
using LinearAlgebra, Printf
DEV = true

function find_local_edkit(start = pwd())
    dir = abspath(start)
    while true
        candidate = joinpath(dir, "src", "EDKit.jl")
        isfile(candidate) && return candidate
        parent = dirname(dir)
        parent == dir && error("Could not locate src/EDKit.jl from $(start)")
        dir = parent
    end
end

if DEV
    include(find_local_edkit())
    using .EDKit
else
    using EDKit
end
using ITensors, ITensorMPS

const HAS_PLOTS = try
    @eval using Plots
    true
catch
    false
end


# XX-chain Majorana/string benchmark with fDAOE

This notebook uses a free-fermion style test where `fdaoe` is the natural truncation. In the Jordan-Wigner language, a local Majorana operator evolves into a sum of string operators of the form `Z ... Z X` and `Z ... Z Y`. The fermionic DAOE MPO is designed to preserve that structure much better than the plain `daoe` filter.

We benchmark this on a short open XX chain, again starting from `X_1` on a chain of length `L = 5`, and compare:

1. exact evolution in the full operator space,
2. TEBD with plain `daoe`,
3. TEBD with `fdaoe`.


In [ ]:
function normalized_overlap(v, w)
    abs(dot(v, w)) / (norm(v) * norm(w))
end

function site_weight_masks(L)
    masks = [falses(4^L) for _ in 1:L]
    for i in 1:4^L
        d = digits(i - 1, base = 4, pad = L)
        for j in 1:L
            masks[j][i] = d[L - j + 1] != 0
        end
    end
    masks
end

function site_weight_profile(v, masks)
    nrm = sum(abs2, v)
    [sum(abs2, v[m]) / nrm for m in masks]
end

bond_dims(psi) = [linkdim(psi, b) for b in 1:length(psi)-1]


In [ ]:
L = 5
dt = 0.05
steps = 20
gamma = 0.3
ps = siteinds("Pauli", L)
B = TensorBasis(L = L, base = 4)
masks = site_weight_masks(L)

h2 = spin((1.0, "xx"), (1.0, "yy"))
superop = commutation_mat(h2)
K = operator(fill(superop, L - 1), [[i, i + 1] for i in 1:L-1], B)
Uexact = exp(dt * Array(K))
gates = tebd4(fill(superop, L - 1), ps, dt)

D = daoe(ps, 2, gamma)
FD = fdaoe(ps, 2, gamma)

v_exact = mps2vec(productMPS(ps, ["X", fill("I", L - 1)...]))
psi_daoe = vec2mps(v_exact, ps)
psi_fdaoe = copy(psi_daoe)


In [ ]:
function run_xx_benchmark(v_exact, psi_daoe, psi_fdaoe, Uexact, gates, D, FD, masks, dt, steps)
    times = collect(0:dt:steps * dt)
    overlap_daoe = zeros(length(times))
    overlap_fdaoe = zeros(length(times))
    daoe_maxbond = zeros(Int, length(times))
    fdaoe_maxbond = zeros(Int, length(times))
    profiles_exact = Matrix{Float64}(undef, length(times), length(masks))
    profiles_daoe = Matrix{Float64}(undef, length(times), length(masks))
    profiles_fdaoe = Matrix{Float64}(undef, length(times), length(masks))

    for n in eachindex(times)
        v_daoe = mps2vec(psi_daoe)
        v_fdaoe = mps2vec(psi_fdaoe)

        overlap_daoe[n] = normalized_overlap(v_exact, v_daoe)
        overlap_fdaoe[n] = normalized_overlap(v_exact, v_fdaoe)
        daoe_maxbond[n] = maximum(bond_dims(psi_daoe); init = 1)
        fdaoe_maxbond[n] = maximum(bond_dims(psi_fdaoe); init = 1)

        profiles_exact[n, :] .= site_weight_profile(v_exact, masks)
        profiles_daoe[n, :] .= site_weight_profile(v_daoe, masks)
        profiles_fdaoe[n, :] .= site_weight_profile(v_fdaoe, masks)

        n == length(times) && break
        v_exact = Uexact * v_exact

        psi_daoe = apply(gates, psi_daoe)
        psi_daoe = apply(D, psi_daoe)
        normalize!(psi_daoe)

        psi_fdaoe = apply(gates, psi_fdaoe)
        psi_fdaoe = apply(FD, psi_fdaoe)
        normalize!(psi_fdaoe)
    end

    (; times, overlap_daoe, overlap_fdaoe, daoe_maxbond, fdaoe_maxbond, profiles_exact, profiles_daoe, profiles_fdaoe)
end

res = run_xx_benchmark(v_exact, psi_daoe, psi_fdaoe, Uexact, gates, D, FD, masks, dt, steps)
times = res.times
overlap_daoe = res.overlap_daoe
overlap_fdaoe = res.overlap_fdaoe
daoe_maxbond = res.daoe_maxbond
fdaoe_maxbond = res.fdaoe_maxbond
profiles_exact = res.profiles_exact
profiles_daoe = res.profiles_daoe
profiles_fdaoe = res.profiles_fdaoe


## Fidelity and bond-dimension comparison

For this free-fermion benchmark, `fdaoe` should stay extremely close to the exact answer while plain `daoe` loses fidelity much earlier. That is the main physics reason to prefer the fermionic filter for Jordan-Wigner string growth.


In [ ]:
if HAS_PLOTS
    p1 = plot(times, overlap_daoe; label = "daoe", xlabel = "time", ylabel = "overlap with exact", ylim = (0.85, 1.01))
    plot!(p1, times, overlap_fdaoe; label = "fdaoe")
    p2 = plot(times, daoe_maxbond; label = "daoe", xlabel = "time", ylabel = "max bond dimension")
    plot!(p2, times, fdaoe_maxbond; label = "fdaoe")
    plot(p1, p2; layout = (1, 2), size = (900, 320))
else
    (; times, overlap_daoe, overlap_fdaoe, daoe_maxbond, fdaoe_maxbond)
end


## Site-resolved operator profile at the final time

This profile again measures the Hilbert-Schmidt weight of strings that are non-identity on each site. Because the exact XX-chain evolution remains close to a fermionic string ansatz, the `fdaoe` profile should track the exact front much more closely than the plain `daoe` profile.


In [ ]:
final_profile = (; 
    site = collect(1:L),
    exact = vec(profiles_exact[end, :]),
    daoe = vec(profiles_daoe[end, :]),
    fdaoe = vec(profiles_fdaoe[end, :]),
)

if HAS_PLOTS
    plot(final_profile.site, final_profile.exact; label = "exact", marker = :circle, xlabel = "site", ylabel = "operator weight", ylim = (0, 1.05))
    plot!(final_profile.site, final_profile.daoe; label = "daoe", marker = :square)
    plot!(final_profile.site, final_profile.fdaoe; label = "fdaoe", marker = :diamond)
else
    final_profile
end


At this small size, the difference is already clear: `fdaoe` preserves the exact free-fermion operator growth almost perfectly at the same cutoff where plain `daoe` noticeably distorts it. That is the simplest compact benchmark I know for why the fermionic version exists.
